# 08.05 Production Monitoring — 관측 · 알림 · PII

프로덕션은 "트레이스 한 번 잘 찍어보자" 가 아니라 **실시간 대시보드 + 자동 평가 + 알림 + 개인정보 방어** 가 같이 돌아야 한다. 이 노트북은 배포 후에 필요한 LangSmith 기능을 운영 관점에서 묶는다.

## 학습 목표

- 프로젝트 대시보드(Overview / Analytics) 에서 latency p50/p95, cost, 성공률을 읽는다
- Online evaluator 를 자동 실행되는 autoeval rule 로 등록한다
- 앱에서 `client.create_feedback(run_id, ...)` 로 사용자 thumbs/별점을 전송한다
- Metadata 기반 알림 룰(실패율 > N% -> webhook) 을 이해한다
- 고볼륨 프로덕션에서 `LANGSMITH_TRACING_SAMPLING_RATE` 로 샘플링한다
- PII 스크러빙 — LangChain `PIIMiddleware` 와 LangSmith `hide_inputs`/`anonymizer` 조합
- Slack / PagerDuty 웹훅 수신 패턴

## 사전 준비

```dotenv
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=langsmith-prod-demo
# 선택: 고볼륨 환경에서 10% 만 트레이스
# LANGSMITH_TRACING_SAMPLING_RATE=0.1
# 선택: 전체 입력/출력 차단
# LANGSMITH_HIDE_INPUTS=true
# LANGSMITH_HIDE_OUTPUTS=true
OPENAI_API_KEY=sk-...
```

프로덕션에서는 이 노트북의 예제 프로젝트(`langsmith-prod-demo`) 처럼 **환경별로 프로젝트를 분리**해 두는 것이 기본이다.

In [ ]:
# !pip install -U langsmith langchain langchain-openai

from dotenv import load_dotenv
import os
load_dotenv(override=True)

assert os.environ.get("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY 미설정"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 미설정"
os.environ.setdefault("LANGSMITH_PROJECT", "langsmith-prod-demo")
print("프로젝트:", os.environ["LANGSMITH_PROJECT"])
print("sampling rate:", os.environ.get("LANGSMITH_TRACING_SAMPLING_RATE", "1.0 (전체)"))

## 8.05.1 프로젝트 대시보드 — latency · cost · 성공률

UI 의 프로젝트 페이지에는 기본 3개 탭이 있다.

| 탭 | 보는 것 | 알림 연결 |
|----|---------|-----------|
| **Runs** | trace 목록, 필터, 빠른 검색 | — |
| **Monitor** | latency p50·p95·p99, 성공률, error 분포, token/cost 시계열 | Rules 로 임계 알림 |
| **Evaluators** | 부착된 online evaluator 와 score 분포 | 특정 key 점수 하락 알림 |

대시보드를 **직접 만들고 싶다면** `Dashboards` 에서 커스텀 차트를 묶을 수 있다. 같은 지표를 `client.list_runs` 로 뽑아 Grafana 같은 사내 시스템에 붙여도 된다.

<!-- phase-c:embed -->
![Project Monitoring Dashboards](../assets/images/langsmith/05_production_monitoring/01_monitoring_dashboards_list.png)

*Monitoring > Dashboards — 6개 탭(Traces / LLM Calls / Cost & Tokens / Tools / Run Types / Feedback Scores) × 기간(Last 7 days 기본). 위 그림에서 Fri 17 spike 는 이 curriculum 실행분이 한 번에 몰리며 발생한 pattern. Trace Count / Latency / Error Rate / LLM Count / LLM Latency 모두 time-series 로 표시.*

In [ ]:
# /runs/query 의 단일 요청 limit 은 서버 측에서 100 으로 제한된다.
# list_runs 는 generator 라 itertools.islice 로 N 개까지 자동 페이지네이션.
from itertools import islice
from langsmith import Client
import statistics

client = Client()
runs = list(islice(
    client.list_runs(project_name=os.environ["LANGSMITH_PROJECT"]),
    500,   # 서버 cap 100 을 넘어 500 개까지 자동으로 모음
))

durations = [
    (r.end_time - r.start_time).total_seconds()
    for r in runs if r.end_time and r.start_time
]
if durations:
    print(f"표본 {len(durations)}건 — p50={statistics.median(durations):.2f}s "
          f"p95={statistics.quantiles(durations, n=20)[-1]:.2f}s "
          f"p99={statistics.quantiles(durations, n=100)[-1]:.2f}s")

## 8.05.2 Online evaluator — autoeval rule

`03_datasets_and_evaluation` 에서 UI 로 붙이는 흐름을 봤다면, 여기선 **운영 관점 설계**다.

| 역할 | 설정 예 |
|------|---------|
| 실시간 품질 게이지 | `has(tags, "env:prod")` 인 run 에 LLM-as-judge(`useful` score 0~1) |
| 비용 관리 | Sampling rate 0.05 로 5% 만 평가 |
| 회귀 감지 | `useful` score 평균이 N% 이하로 떨어지면 Rules 에서 webhook 트리거 |
| 특정 유즈케이스 | `metadata.feature == "checkout"` 인 run 에만 평가 붙이기 |

autoeval 결과는 feedback key 로 저장되므로, 다음 셀의 알림 룰 · 대시보드 · Experiments 비교에 모두 재사용된다.

<!-- phase-c:embed -->
![Automations 탭](../assets/images/langsmith/05_production_monitoring/03_automations_tab.png)

*프로젝트 > Automations 탭 — `No automations found` 상태. `+ Automation` 으로 규칙 정의: 조건(ex. Feedback score < 0.5) → 액션(ex. dataset 으로 이관 / 웹훅 호출 / annotation queue 로 전송).*

## 8.05.3 사용자 feedback 수집 API

앱 UI 의 thumbs-up / 별점 / "이 답 도움 안 됨" 버튼 -> 서버에서 `client.create_feedback` 호출이 정석 패턴. 클라이언트가 기다리지 않도록 **background** 로 쏜다 (Python SDK 는 `trace_id` 를 주면 자동 백그라운드).

In [ ]:
# 애플리케이션 내부 예시: 에이전트 실행 후 run id 를 캡처해 돌려주고,
# 사용자 피드백이 오면 그 id 로 feedback 을 붙인다.
from langsmith import traceable

@traceable(name="answer-user")
def answer_user(question: str) -> str:
    # 실제로는 agent.invoke(...) 결과를 반환
    return f"'{question}' 에 대한 답변."

# 실행 시 run_tree 로 현재 run_id 를 가져오는 예
from langsmith.run_helpers import get_current_run_tree

@traceable(name="answer-with-id")
def answer_with_id(question: str):
    rt = get_current_run_tree()
    return {"answer": answer_user(question), "run_id": str(rt.id)}

reply = answer_with_id("서울 날씨?")
print(reply)

# 나중에 사용자가 thumbs-up 을 누르면
client.create_feedback(
    run_id=reply["run_id"],
    key="user_thumbs",
    score=1,
    comment="도움이 됐다고 응답",
)
print("피드백 전송 완료")

## 8.05.4 Metadata 기반 알림 룰 — 실패율 > N% -> webhook

UI 의 프로젝트 -> **Rules** 탭에서 다음 액션들을 조합한다.

- `Add to annotation queue` (사람 리뷰)
- `Add to dataset` (골든셋 축적)
- `Trigger webhook` (Slack, PagerDuty, 자체 incident 시스템)
- `Extend data retention` (중요 trace 보존 연장)
- `Run online evaluator` (조건부 품질 평가)

**필터 표현식 예**

```
and(
  has(tags, "env:prod"),
  eq(status, "error")
)
```

액션 실행 순서(LangSmith 고정): annotation queue -> dataset -> webhook -> online evaluator -> code evaluator -> alert. 샘플링 비율도 룰 단위로 다르게 줄 수 있다 (예: 0.5 = 매칭의 50%).

<!-- phase-c:embed -->
![Alerts 리스트](../assets/images/langsmith/05_production_monitoring/02_alerts_page.png)

*Monitoring > Alerts — 조직 전체 알림 규칙 허브. `Create Alert` 클릭 시 모달:*

![Create Alert 모달](../assets/images/langsmith/05_production_monitoring/05_create_alert_modal.png)

*Tracing project 선택 후 조건(에러율/지연/피드백 변화) 지정 → webhook URL 로 전송 설정.*

## 8.05.5 고볼륨 샘플링 — `LANGSMITH_TRACING_SAMPLING_RATE`

초당 수백 요청이 되면 모든 trace 를 보내는 건 비용·네트워크 낭비다. 두 층으로 제어한다.

| 층위 | 수단 | 특성 |
|------|------|------|
| 프로세스 전체 | `LANGSMITH_TRACING_SAMPLING_RATE=0.1` | `@traceable`, `RunTree`, 자동 계측 모두 적용 |
| 요청별 | `Client(tracing_sampling_rate=...)` + `tracing_context` | 관리자/결제 등 중요 요청만 100% |

조합하면 "일반 트래픽은 10%, 결제 트래픽은 100%" 같은 운영 정책을 구현할 수 있다.

In [ ]:
from langsmith import Client
from langsmith.run_helpers import tracing_context

# 기본 클라이언트는 10% 만, 중요 흐름용 클라이언트는 100% 기록
client_bulk     = Client(tracing_sampling_rate=0.1)
client_critical = Client(tracing_sampling_rate=1.0)

with tracing_context(client=client_critical, tags=["env:prod", "flow:checkout"]):
    # 결제 같은 핵심 흐름은 이 블록 안에서 실행 — 모두 기록
    print("critical 흐름 — 100% 트레이스")

with tracing_context(client=client_bulk, tags=["env:prod", "flow:chat"]):
    # 일반 잡담 — 10% 만 샘플링
    print("bulk 흐름 — 10% 샘플링")

## 8.05.6 PII 스크러빙 — `PIIMiddleware` + `anonymizer`

두 레이어로 방어한다.

1. **모델 입력 단계**: `langchain.agents.middleware.PIIMiddleware` 가 LLM 에 전달되는 메시지에서 이메일·카드번호·API 키를 차단/마스킹 — 모델 자체가 PII 를 보지 않게 한다.
2. **트레이스 전송 단계**: LangSmith 클라이언트의 `hide_inputs`/`hide_outputs` 또는 `anonymizer` 가 서버로 올라가는 input/output 에서 한 번 더 씻어낸다.

둘 다 걸어야 "모델에 들어갔지만 트레이스엔 안 남는" 혹은 "모델은 못 봤지만 trace 에 남았다" 같은 구멍이 없다.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool

@tool
def lookup_order(order_id: str) -> str:
    """주문 상태 조회 (데모용)."""
    return f"{order_id} -> 배송 중"

# 1) 모델 입력 단계 — PIIMiddleware 로 마스킹
# langchain 1.2.x 의 PIIMiddleware 는 기본 pii_type 이 email/credit_card/ip/mac_address/url 5종.
# "api_key" 같은 커스텀 타입을 쓰려면 detector 를 직접 제공해야 한다.
safe_agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[lookup_order],
    middleware=[
        PIIMiddleware("email",       strategy="redact"),
        PIIMiddleware("credit_card", strategy="redact"),
    ],
)

out = safe_agent.invoke({"messages": [
    {"role": "user", "content": "내 이메일 alice@example.com 주문 상태 확인해 줘 (주문 A-42)"}
]})
print(out["messages"][-1].content)

In [ ]:
# 2) 트레이스 전송 단계 — anonymizer 로 input/output 재마스킹
from langsmith.anonymizer import create_anonymizer

anonymizer = create_anonymizer([
    {"pattern": r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", "replace": "<email>"},
    {"pattern": r"\b\d{4}[ -]?\d{4}[ -]?\d{4}[ -]?\d{4}\b",            "replace": "<card>"},
])

masked_client = Client(anonymizer=anonymizer)

# 전체 차단이 필요하면 환경변수 또는 Client 인자로
fully_hidden_client = Client(
    hide_inputs=lambda inputs: {},
    hide_outputs=lambda outputs: {},
)
print("anonymizer / full-hide client 준비 완료 — Client 인스턴스를 바꿔 주입")

## 8.05.7 Slack / PagerDuty 웹훅 연동

Rules 의 webhook 액션은 설정한 URL 로 POST 페이로드를 쏜다. 받는 쪽에서 Slack/PagerDuty 포맷으로 변환해 알림을 띄운다.

```python
# 예: FastAPI 로 받는 webhook — Slack 으로 중계
from fastapi import FastAPI, Request
import httpx, os

app = FastAPI()
SLACK_URL     = os.environ["SLACK_INCOMING_WEBHOOK"]
PAGERDUTY_URL = os.environ.get("PAGERDUTY_EVENTS_V2_URL")

@app.post("/langsmith/alert")
async def on_alert(req: Request):
    event = await req.json()
    run_id  = event.get("run_id") or event.get("trace_id")
    status  = event.get("status", "unknown")
    tags    = event.get("tags", [])
    trace_url = f"https://smith.langchain.com/o/-/projects/-/r/{run_id}"

    async with httpx.AsyncClient() as hc:
        # Slack
        await hc.post(SLACK_URL, json={
            "text": f":rotating_light: LangSmith alert — status={status} tags={tags}\n<{trace_url}|trace 열기>",
        })
        # PagerDuty (심각도 높을 때만)
        if PAGERDUTY_URL and "critical" in tags:
            await hc.post(PAGERDUTY_URL, json={
                "routing_key": os.environ["PAGERDUTY_ROUTING_KEY"],
                "event_action": "trigger",
                "payload": {"summary": f"LangSmith {status}", "severity": "error", "source": "langsmith"},
            })
    return {"ok": True}
```

UI 의 Rules 에서 이 엔드포인트 URL 을 webhook 대상으로 등록하고, 필터를 `and(has(tags, "env:prod"), eq(status, "error"))` 로 잡으면 프로덕션 에러만 Slack 으로 뜬다.

## 체크리스트

- [ ] 프로젝트 Monitor 탭에서 p50/p95/p99 · 성공률 · cost 확인
- [ ] Online evaluator 를 prod 태그 필터 + 5% 샘플링으로 등록
- [ ] 앱에서 `client.create_feedback(run_id, ...)` 로 thumbs 전송 패턴 구현
- [ ] Rules 로 `status=error` + 태그 필터에 webhook 액션 부착
- [ ] `LANGSMITH_TRACING_SAMPLING_RATE` 와 `tracing_context` 로 중요 흐름만 100% 샘플링
- [ ] `PIIMiddleware`(모델) + `anonymizer`/`hide_inputs`(트레이스) 이중 방어
- [ ] Webhook 수신 서비스가 Slack / PagerDuty 로 라우팅하는지 확인

## 다음

- 이걸로 `08_langsmith` 커리큘럼은 마무리. Langfuse·OTel 같은 벤더 중립 관측과 비교는 `07_integration/12_observability/` 에서 이어진다.

## 참고

- Observability 전체: https://docs.langchain.com/langsmith/observability
- Rules · webhook: https://docs.langchain.com/langsmith/rules
- Sampling: https://docs.langchain.com/langsmith/sample-traces
- Mask inputs/outputs: https://docs.langchain.com/langsmith/mask-inputs-outputs
- Feedback API: https://docs.langchain.com/langsmith/attach-user-feedback

<!-- phase-c:embed -->
## 8.05.X Insights Agent (유료)

![Insights 탭 — Upgrade required](../assets/images/langsmith/05_production_monitoring/04_insights_tab.png)

*프로젝트 > Insights 탭 — LangSmith 의 **Insights Agent** 로 production trace 에서 usage pattern / common failure modes 를 자동 추출. 무료 플랜은 upgrade 필요.*